## 🌍 Extracción y Clasificación de Lugares con OpenStreetMap (OSM)

En esta sección se configuran los tipos de lugares que se desean extraer desde **OpenStreetMap** con la ayuda de la librería `osmnx`, y se implementa una clase `ExtraccionOSM` que permite automatizar la descarga, procesamiento y clasificación funcional de dichos puntos de interés.

---

### 🔧 Configuración inicial

- Se define el path raíz del proyecto para importar configuraciones externas.
- Se importa el path `DATA_EXTERNAL_OSM_DIR`, donde se guardarán los resultados.
- Se establece un diccionario `tipos_osm` con categorías relevantes de OSM: `amenity`, `shop`, `leisure`, `transport`, etc.

### 🗂️ Traducción y agrupación funcional

- Se crea un diccionario `traduccion_tipo` para traducir los códigos de OSM al español.
- Se define `grupo_funcional`, que agrupa lugares por su rol en el entorno:
  - `educ`: educación (escuelas, universidades, etc.)
  - `salud`: salud (hospitales, farmacias, etc.)
  - `com_serv`: comercios y servicios básicos
  - `aliment_ocio`: restaurantes, cafés, bares
  - `cult_recre`: cultura y recreación
  - `transp_pub`: transporte público
  - `zona_urb`: zonas industriales y comerciales
  - `entorno_nat`: naturaleza (parques, playas, agua)

---

### 🧠 Clase `ExtraccionOSM`

Esta clase tiene tres métodos clave:

- `__init__`: Define la ruta de salida del archivo.
- `obtener_lugares(ciudad, filtros_osm)`: Descarga los lugares que coinciden con los filtros definidos desde el polígono geográfico de la ciudad.
- `procesar_y_guardar(lugares)`: 
  - Filtra solo los lugares con coordenadas (tipo `Point`).
  - Traduce y agrupa cada tipo de lugar según su categoría funcional.
  - Guarda los resultados en un archivo `.xlsx`.

---

### ✅ Ejecución

- Se define `ruta_salida` y se instancia la clase.
- Se extraen los lugares de “Melbourne, Australia”.
- Se procesa y guarda el archivo en Excel con columnas como:
  - `lat`, `lon`: coordenadas
  - `tipo_osm`: código original (ej. `school`)
  - `tipo`: traducción al español (ej. `Escuela`)
  - `grupo`: grupo funcional (ej. `educ`)


In [57]:
import os
import sys
import pandas as pd
import osmnx as ox

# ============
# CONFIGURACIÓN
# ============

# Ruta base si usas config externa
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from config import DATA_EXTERNAL_OSM_DIR

# Tipos de lugares que queremos extraer de OSM
tipos_osm = {
    'amenity': [
        'school', 'kindergarten', 'university', 'college', 'hospital', 'clinic', 'pharmacy',
        'library', 'restaurant', 'cafe', 'bar', 'bank', 'atm', 'supermarket', 'convenience',
        'place_of_worship', 'theatre', 'cinema', 'museum', 'community_centre', 'childcare'
    ],
    'shop': True,
    'leisure': ['park', 'playground', 'sports_centre', 'pitch'],
    'public_transport': True,
    'railway': ['station', 'subway_entrance'],
    'highway': ['bus_stop'],
    'natural': ['wood', 'water', 'beach'],
    'landuse': ['industrial', 'commercial']
}

# Traducción de tipos al español
traduccion_tipo = {
    'station': 'Estación',
    'stop_position': 'Paradero',
    'bus_stop': 'Paradero de bus',
    'subway_entrance': 'Entrada de metro',
    'school': 'Escuela',
    'kindergarten': 'Jardín',
    'university': 'Universidad',
    'college': 'Instituto',
    'childcare': 'Guardería',
    'hospital': 'Hospital',
    'clinic': 'Clínica',
    'pharmacy': 'Farmacia',
    'doctors': 'Consultorio',
    'supermarket': 'Supermercado',
    'convenience': 'Tienda',
    'restaurant': 'Restaurante',
    'cafe': 'Café',
    'bar': 'Bar',
    'place_of_worship': 'Templo',
    'library': 'Biblioteca',
    'cinema': 'Cine',
    'theatre': 'Teatro',
    'museum': 'Museo',
    'park': 'Parque',
    'playground': 'Juegos',
    'sports_centre': 'Deporte',
    'pitch': 'Cancha',
    'bank': 'Banco',
    'atm': 'Cajero',
    'community_centre': 'Centro comunitario',
    'wood': 'Bosque',
    'water': 'Agua',
    'beach': 'Playa',
    'industrial': 'Zona industrial',
    'commercial': 'Zona comercial'
}

# Grupos generales
# grupo_funcional = {
#     'Educación': ['school', 'kindergarten', 'university', 'college', 'childcare'],
#     'Salud': ['hospital', 'clinic', 'pharmacy', 'doctors'],
#     'Comercio': ['shop', 'supermarket', 'convenience'],
#     'Restauración': ['restaurant', 'cafe', 'bar'],
#     'Religioso': ['place_of_worship', 'community_centre'],
#     'Cultura y recreación': ['library', 'cinema', 'theatre', 'museum', 'park', 'playground', 'sports_centre', 'pitch'],
#     'Transporte': ['station', 'bus_stop', 'subway_entrance', 'stop_position'],
#     'Financiero': ['bank', 'atm'],
#     'Zona urbana': ['industrial', 'commercial'],
#     'Naturaleza': ['wood', 'water', 'beach']
# }

grupo_funcional = {
    'educ': [  # Educación
        'school', 'kindergarten', 'university', 'college', 'childcare'
    ],
    'salud': [  # Servicios de salud
        'hospital', 'clinic', 'pharmacy', 'doctors'
    ],
    'com_serv': [  # Comercio y servicios básicos
        'shop', 'supermarket', 'convenience'
    ],
    'finenciero': [  # financiero
        'bank', 'atm'
    ],
    'aliment_ocio': [  # Alimentación y ocio
        'restaurant', 'cafe', 'bar'
    ],
    'cult_recre': [  # Cultura y recreación
        'library', 'cinema', 'theatre', 'museum', 'park', 'playground', 'sports_centre', 'pitch'
    ],
    'transp_pub': [  # Transporte público
        'station', 'bus_stop', 'subway_entrance', 'stop_position'
    ],
    'social_com': [  # Centros sociales o comunitarios
        'place_of_worship', 'community_centre'
    ],
    'zona_urb': [  # Zonas urbanas (comercio/industria)
        'industrial', 'commercial'
    ],
    'entorno_nat': [  # Naturaleza o entorno ambiental
        'wood', 'water', 'beach'
    ]
}



# ============
# EXTRACTOR
# ============

class ExtraccionOSM:
    def __init__(self, ruta_excel):
        self.ruta_excel = ruta_excel
        os.makedirs(os.path.dirname(self.ruta_excel), exist_ok=True)

    def obtener_lugares(self, ciudad, filtros_osm):
        zona = ox.geocode_to_gdf(ciudad)
        poligono = zona.geometry.iloc[0]
        lugares = ox.features_from_polygon(poligono, tags=filtros_osm)
        return lugares

    def procesar_y_guardar(self, lugares):
        registros = []

        columnas_utiles = [col for col in lugares.columns if col in [
            'amenity', 'shop', 'leisure', 'public_transport',
            'railway', 'highway', 'natural', 'landuse'
        ]]

        for col in columnas_utiles:
            sub = lugares[[col, 'geometry']].dropna()
            sub = sub[sub.geometry.type == 'Point']
            for _, row in sub.iterrows():
                tipo_en = row[col]
                tipo_es = traduccion_tipo.get(tipo_en, tipo_en)
                grupo = self.obtener_grupo(tipo_en)

                registros.append({
                    'lat': row.geometry.y,
                    'lon': row.geometry.x,
                    'tipo_osm': tipo_en,
                    'etiqueta': col,
                    'tipo': tipo_es,
                    'grupo': grupo
                })

        df = pd.DataFrame(registros).drop_duplicates()
        df.to_excel(self.ruta_excel, index=False)
        print(f"✅ Archivo guardado en {self.ruta_excel}")

    def obtener_grupo(self, tipo):
        for grupo, lista in grupo_funcional.items():
            if tipo in lista:
                return grupo
        return 'Otro'


# ============
# USO
# ============

ruta_salida = os.path.join(DATA_EXTERNAL_OSM_DIR, 'OpenStreetMap.xlsx')
osm = ExtraccionOSM(ruta_excel=ruta_salida)

data_osm = osm.obtener_lugares("Melbourne, Australia", filtros_osm=tipos_osm)
osm.procesar_y_guardar(data_osm)

✅ Archivo guardado en c:\Users\carlo\OneDrive\U\1er\introds\proyectofinal\data\external\osm_places\OpenStreetMap.xlsx


In [ ]:
import pandas as pd
osm_places = pd.read_excel(os.path.join(DATA_EXTERNAL_OSM_DIR, 'OpenStreetMap.xlsx'))

In [ ]:
osm_places.to_excel(os.path.join(DATA_EXTERNAL_OSM_DIR, 'OpenStreetMap.xlsx'), index=False)